In [ ]:
print("[cell] setup", flush=True)
import subprocess, sys, os
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || true")
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.cuda.device_count(), flush=True)
PIP = os.environ.get("PIP_SPEC", '')   # --env PIP_SPEC=... / PIP_POST=... override the built-in install
if PIP and torch.cuda.is_available():
    sh(f"pip install -q {PIP}")
    if os.environ.get("PIP_POST", ''):
        sh(os.environ.get("PIP_POST", ''))
sh("pip list 2>/dev/null | grep -i -E '^(vllm|torch|transformers|qwen-asr|pandas|openpyxl) ' || true")


In [ ]:
print("[cell] write code files", flush=True)
FILES = {
    'common.py': '"""Shared helpers for GreenScan training on Kaggle (T4) and on a rented A100 (vast.ai).\n\nData is looked up in $GS_DATA, ./data, ../data, then /kaggle/input (a private dataset). Outputs go to $GS_OUT\n(default ./out). Every script prints "[cell] ..." at each stage and flushes, so a log shows where a run is.\n"""\nfrom __future__ import annotations\n\nimport json\nimport os\nimport random\nimport re\nimport time\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\n\nimport numpy as np\n\n\ndef cell(name: str) -> None:\n    print(f"[cell] {name}", flush=True)\n\n\ndef log(*a) -> None:\n    print(*a, flush=True)\n\n\ndef find(name: str) -> Path:\n    for root in (os.environ.get("GS_DATA", ""), "data", "../data", "/kaggle/input"):\n        if root and Path(root).exists():\n            hit = next(iter(sorted(Path(root).rglob(name))), None)\n            if hit:\n                return hit\n    raise FileNotFoundError(name)\n\n\ndef has(name: str) -> bool:\n    try:\n        find(name)\n        return True\n    except FileNotFoundError:\n        return False\n\n\ndef jl(path) -> list[dict]:\n    p = Path(path) if not isinstance(path, str) or Path(path).exists() else find(path)\n    return [json.loads(x) for x in open(p, encoding="utf-8") if x.strip()]\n\n\ndef wjl(path, rows) -> None:\n    Path(path).parent.mkdir(parents=True, exist_ok=True)\n    with open(path, "w", encoding="utf-8") as fh:\n        for r in rows:\n            fh.write(json.dumps(r, ensure_ascii=False) + "\\n")\n\n\ndef out_dir(sub: str = "") -> Path:\n    p = Path(os.environ.get("GS_OUT", "out")) / sub\n    p.mkdir(parents=True, exist_ok=True)\n    return p\n\n\ndef save_json(path, obj) -> None:\n    Path(path).parent.mkdir(parents=True, exist_ok=True)\n    Path(path).write_text(json.dumps(obj, ensure_ascii=False, indent=1), encoding="utf-8")\n\n\ndef seed_all(seed: int) -> None:\n    random.seed(seed)\n    np.random.seed(seed)\n    try:\n        import torch\n        torch.manual_seed(seed)\n        torch.cuda.manual_seed_all(seed)\n    except ImportError:\n        pass\n\n\ndef norm60(s: str) -> str:\n    return re.sub(r"\\s+", " ", (s or "").lower()).strip()[:60]\n\n\ndef company_folds(groups: list[str], k: int = 5, seed: int = 13) -> list[tuple[list[int], list[int]]]:\n    """Out-of-company folds: every company sits in exactly one test fold; folds balanced by row count."""\n    count = Counter(groups)\n    comps = sorted(count, key=lambda c: (-count[c], c))\n    rng = random.Random(seed)\n    rng.shuffle(comps)\n    comps.sort(key=lambda c: -count[c])\n    load, fold_of = [0] * k, {}\n    for c in comps:\n        f = min(range(k), key=lambda i: load[i])\n        fold_of[c] = f\n        load[f] += count[c]\n    out = []\n    for f in range(k):\n        test = [i for i, g in enumerate(groups) if fold_of[g] == f]\n        train = [i for i, g in enumerate(groups) if fold_of[g] != f]\n        if test and train:\n            out.append((train, test))\n    return out\n\n\ndef prf(gold: list[str], pred: list[str], labels: list[str]) -> dict:\n    per = {}\n    for l in labels:\n        tp = sum(g == l and p == l for g, p in zip(gold, pred))\n        fp = sum(g != l and p == l for g, p in zip(gold, pred))\n        fn = sum(g == l and p != l for g, p in zip(gold, pred))\n        pr = tp / (tp + fp) if tp + fp else 0.0\n        rc = tp / (tp + fn) if tp + fn else 0.0\n        per[l] = {"precision": round(pr, 4), "recall": round(rc, 4),\n                  "f1": round(2 * pr * rc / (pr + rc), 4) if pr + rc else 0.0, "support": tp + fn, "predicted": tp + fp}\n    present = [l for l in labels if per[l]["support"]]\n    macro = sum(per[l]["f1"] for l in present) / len(present) if present else 0.0\n    acc = sum(g == p for g, p in zip(gold, pred)) / len(gold) if gold else 0.0\n    return {"macro_f1": round(macro, 4), "accuracy": round(acc, 4), "per_class": per, "n": len(gold)}\n\n\ndef confusion(gold: list[str], pred: list[str], labels: list[str]) -> list[list[int]]:\n    idx = {l: i for i, l in enumerate(labels)}\n    m = [[0] * len(labels) for _ in labels]\n    for g, p in zip(gold, pred):\n        if g in idx and p in idx:\n            m[idx[g]][idx[p]] += 1\n    return m\n\n\ndef boot_ci(items: list, groups: list[str], stat, n: int = 1000, seed: int = 7) -> list[float] | None:\n    """95% interval of stat(items) resampling whole companies (cluster bootstrap)."""\n    by = defaultdict(list)\n    for it, g in zip(items, groups):\n        by[g].append(it)\n    keys = sorted(by)\n    if len(keys) < 2:\n        return None\n    rng = random.Random(seed)\n    vals = []\n    for _ in range(n):\n        sample = [it for _ in keys for it in by[rng.choice(keys)]]\n        try:\n            vals.append(stat(sample))\n        except ZeroDivisionError:\n            continue\n    vals.sort()\n    return [round(vals[int(0.025 * len(vals))], 4), round(vals[int(0.975 * len(vals)) - 1], 4)] if vals else None\n\n\nclass Timer:\n    def __init__(self):\n        self.t0 = time.time()\n\n    def __call__(self) -> float:\n        return round(time.time() - self.t0, 1)\n',
    'train_stance_enc.py': '"""Offline stance model (CPU): mDeBERTa-v3-base-xnli fine-tuned as a 4-relation cross-encoder (claim, passage).\n\n    python -u train_stance_enc.py [--epochs 4] [--folds 5]\n\nLabels: pairs_labels.jsonl (Claude A/B + adjudication, CAP v1.0). Pairs whose claim overlaps GS-Eval are dropped so S5\nstays an external test. Loss: class-weighted cross-entropy (∝ 1/sqrt(frequency)); temperature fitted on out-of-fold\nlogits. A relation the labels never contain (CONTRADICTS is almost absent) is never predicted alone: the model abstains\nto CONTEXT unless P(CONTRADICTS) >= 0.9.\nReported (out/stance/metrics.json): out-of-company CV on the pairs (macro-F1, per class, confusion, risk-coverage),\nbaselines on the same pairs (rules, zero-shot NLI), S5 external test (macro-F1 over S and O, PARTIAL counted as S,\nfalse C per 100) against the rules, CPU ms per pair. Model of the final fit: out/stance/mdeberta-stance-final/.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport math\nimport random\nimport sys\nimport time\nfrom collections import Counter\nfrom pathlib import Path\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nfrom common import (Timer, boot_ci, cell, company_folds, confusion, has, jl, log, out_dir, prf,  # noqa: E402\n                    save_json, seed_all)\n\nBASE = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"\nLABELS = ["SUPPORTS", "PARTIAL", "CONTRADICTS", "CONTEXT"]\nL2I = {l: i for i, l in enumerate(LABELS)}\nMAX_LEN, BATCH = 384, 16\nTO_SCO = {"SUPPORTS": "S", "PARTIAL": "S", "CONTRADICTS": "C", "CONTEXT": "O"}\n\n\ndef encode(tok, claims, passages, dev):\n    return tok(claims, passages, truncation="only_second", max_length=MAX_LEN, padding=True, return_tensors="pt").to(dev)\n\n\ndef fit(rows, epochs, lr, seed, hist, tag):\n    import torch\n    from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup\n    seed_all(seed)\n    dev = "cuda" if torch.cuda.is_available() else "cpu"\n    tok = AutoTokenizer.from_pretrained(BASE)\n    model = AutoModelForSequenceClassification.from_pretrained(\n        BASE, num_labels=4, id2label=dict(enumerate(LABELS)), label2id=L2I, ignore_mismatched_sizes=True).float().to(dev)\n    cnt = Counter(L2I[r["relation"]] for r in rows)\n    w = torch.tensor([1 / math.sqrt(cnt[i]) if cnt.get(i) else 0.0 for i in range(4)], device=dev)\n    lossf = torch.nn.CrossEntropyLoss(weight=w * (sum(1 for x in w if x > 0)) / w.sum())\n    steps = epochs * math.ceil(len(rows) / BATCH)\n    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)\n    sch = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)\n    scaler = torch.amp.GradScaler(enabled=dev == "cuda")\n    model.train()\n    for ep in range(epochs):\n        order = list(range(len(rows)))\n        random.shuffle(order)\n        tot = 0.0\n        for i in range(0, len(order), BATCH):\n            b = [rows[j] for j in order[i:i + BATCH]]\n            x = encode(tok, [r["claim"] for r in b], [r["passage"] for r in b], dev)\n            y = torch.tensor([L2I[r["relation"]] for r in b], device=dev)\n            with torch.autocast(dev, dtype=torch.float16, enabled=dev == "cuda"):\n                loss = lossf(model(**x).logits.float(), y)\n            opt.zero_grad()\n            scaler.scale(loss).backward()\n            scaler.unscale_(opt)\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n            scaler.step(opt)\n            scaler.update()\n            sch.step()\n            tot += loss.item() * len(b)\n        hist.append({"run": tag, "epoch": ep + 1, "loss": round(tot / len(rows), 4)})\n        log(f"[stance] {tag} epoch {ep + 1} loss {tot / len(rows):.4f}")\n    return model.eval(), tok\n\n\ndef logits(model, tok, claims, passages):\n    import torch\n    dev = next(model.parameters()).device\n    out = []\n    with torch.no_grad():\n        for i in range(0, len(claims), 32):\n            x = encode(tok, claims[i:i + 32], passages[i:i + 32], dev)\n            out += model(**x).logits.float().cpu().tolist()\n    return out\n\n\ndef softmax(z, t=1.0):\n    m = max(z)\n    e = [math.exp((v - m) / t) for v in z]\n    s = sum(e)\n    return [v / s for v in e]\n\n\ndef decide(p, c_min=0.9):\n    i = max(range(4), key=lambda k: p[k])\n    if LABELS[i] == "CONTRADICTS" and p[i] < c_min:\n        return "CONTEXT"\n    return LABELS[i]\n\n\ndef fit_temperature(z, y):\n    best = (1e9, 1.0)\n    for t in [0.5 + 0.1 * k for k in range(46)]:\n        nll = -sum(math.log(max(softmax(zi, t)[yi], 1e-12)) for zi, yi in zip(z, y)) / len(y)\n        best = min(best, (nll, t))\n    return round(best[1], 2), round(best[0], 4)\n\n\ndef risk_coverage(p, gold):\n    """Accuracy of the most confident fraction, for coverage 10% .. 100%."""\n    order = sorted(range(len(p)), key=lambda i: -max(p[i]))\n    out = []\n    for c in range(1, 11):\n        k = max(1, round(len(order) * c / 10))\n        sel = order[:k]\n        out.append({"coverage": c / 10, "accuracy": round(sum(decide(p[i]) == gold[i] for i in sel) / k, 4),\n                    "min_conf": round(max(p[sel[-1]]), 4)})\n    return out\n\n\ndef sco_scores(gold, pred, groups):\n    def mf(items):\n        g, p = [a for a, _ in items], [b for _, b in items]\n        r = prf(g, p, ["S", "O"])\n        return r["macro_f1"]\n    items = list(zip(gold, pred))\n    return {"macro_f1_SO": round(mf(items), 4), "ci95": boot_ci(items, groups, mf),\n            "false_C": sum(p == "C" and g != "C" for g, p in items), "C_found": sum(p == "C" and g == "C" for g, p in items),\n            "C_gold": sum(g == "C" for g in gold), "false_C_per_100": round(100 * sum(p == "C" and g != "C" for g, p in items) / len(items), 2),\n            "n": len(items), "report": prf(gold, pred, ["S", "C", "O"])}\n\n\ndef main() -> int:\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--epochs", type=int, default=4)\n    ap.add_argument("--lr", type=float, default=2e-5)\n    ap.add_argument("--folds", type=int, default=5)\n    args = ap.parse_args()\n    import torch\n\n    cell("stance data")\n    pool = {r["id"]: r for r in jl("pairs_pool.jsonl")}\n    labels = jl("pairs_labels.jsonl")\n    rows = [{**pool[x["id"]], "relation": x["relation"]} for x in labels\n            if x["id"] in pool and not pool[x["id"]]["overlaps_gs_eval"] and x.get("decided_by") != "contested"]\n    groups = [r["company_id"] for r in rows]\n    log(f"[stance] {len(rows)} labelled pairs · {dict(Counter(r[\'relation\'] for r in rows))} · companies {len(set(groups))}")\n    od = out_dir("stance")\n    hist, m = [], {"n": len(rows), "relations": dict(Counter(r["relation"] for r in rows))}\n\n    cell("stance cv")\n    t = Timer()\n    oof = [None] * len(rows)\n    for f, (tr, te) in enumerate(company_folds(groups, args.folds)):\n        model, tok = fit([rows[i] for i in tr], args.epochs, args.lr, 13, hist, f"fold{f}")\n        for i, z in zip(te, logits(model, tok, [rows[i]["claim"] for i in te], [rows[i]["passage"] for i in te])):\n            oof[i] = z\n        del model\n        torch.cuda.empty_cache()\n    gold = [r["relation"] for r in rows]\n    temp, nll = fit_temperature(oof, [L2I[g] for g in gold])\n    probs = [softmax(z, temp) for z in oof]\n    pred = [decide(p) for p in probs]\n    m["cv"] = {**prf(gold, pred, LABELS), "confusion": confusion(gold, pred, LABELS), "labels": LABELS,\n               "temperature": temp, "nll": nll, "risk_coverage": risk_coverage(probs, gold), "seconds": t(),\n               "ci95_macro_f1": boot_ci(list(zip(gold, pred)), groups,\n                                        lambda it: prf([a for a, _ in it], [b for _, b in it], LABELS)["macro_f1"])}\n    base = {}\n    for name in ("rules", "nli-mdeberta"):\n        if has(f"pairs_cand_{name}.jsonl"):\n            cand = {c["id"]: c["relation"] for c in jl(f"pairs_cand_{name}.jsonl")}\n            bp = [cand.get(r["id"], "CONTEXT") for r in rows]\n            base[name] = {**prf(gold, bp, LABELS), "confusion": confusion(gold, bp, LABELS)}\n    m["cv_baselines"] = base\n    log(f"[stance] CV macro-F1 {m[\'cv\'][\'macro_f1\']} (T={temp}) · baselines "\n        f"{ {k: v[\'macro_f1\'] for k, v in base.items()} }")\n\n    cell("stance final + S5")\n    model, tok = fit(rows, args.epochs, args.lr, 13, hist, "final")\n    model.save_pretrained(od / "mdeberta-stance-final")\n    tok.save_pretrained(od / "mdeberta-stance-final")\n    s5 = jl("s5_pairs.jsonl")\n    z = logits(model, tok, [r["claim"] for r in s5], [r["evidence"] for r in s5])\n    pm = [TO_SCO[decide(softmax(v, temp))] for v in z]\n    pr = [TO_SCO.get(r.get("rules") or "CONTEXT", "O") for r in s5]\n    g5 = [r["gold"] for r in s5]\n    c5 = [r["company"] for r in s5]\n    m["s5"] = {"model": sco_scores(g5, pm, c5), "rules": sco_scores(g5, pr, c5)}\n    log(f"[stance] S5 model {m[\'s5\'][\'model\'][\'macro_f1_SO\']} {m[\'s5\'][\'model\'][\'ci95\']} false C "\n        f"{m[\'s5\'][\'model\'][\'false_C\']} · rules {m[\'s5\'][\'rules\'][\'macro_f1_SO\']} false C {m[\'s5\'][\'rules\'][\'false_C\']}")\n\n    cell("stance cpu latency")\n    cpu = model.to("cpu")\n    torch.set_num_threads(4)\n    t0 = time.time()\n    logits(cpu, tok, [r["claim"] for r in s5[:48]], [r["evidence"] for r in s5[:48]])\n    m["cpu_ms_per_pair_4threads"] = round((time.time() - t0) / 48 * 1000, 1)\n    save_json(od / "metrics.json", m)\n    save_json(od / "history.json", hist)\n    log(f"[stance] cpu {m[\'cpu_ms_per_pair_4threads\']} ms/pair · DONE")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n',
    'train_reranker.py': '"""Offline reranker (CPU): gte-multilingual-reranker-base fine-tuned on the pair relevance labels (2 / 1 / 0).\n\n    python -u train_reranker.py [--epochs 3] [--folds 5]\n\nEach claim of the pool comes with the passages every retriever put in its top 5 (labelled: 2 = an auditor would cite it,\n1 = same subject, 0 = unrelated). Loss: listwise softmax cross-entropy, one relevance-2 passage against up to 7\npassages of relevance 0/1 from the same claim. Out-of-company folds; each held-out claim\'s pooled passages are\nre-ordered by the model and scored: nDCG@5 (gain 2^rel - 1), MRR@5 and hit@5 of a relevance-2 passage, against the\nlite order (offline product), bm25, and the zero-shot reranker. Writes out/reranker/metrics.json and the final model.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport ast\nimport math\nimport random\nimport sys\nimport time\nfrom collections import defaultdict\nfrom pathlib import Path\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nfrom common import Timer, boot_ci, cell, company_folds, jl, log, out_dir, save_json, seed_all  # noqa: E402\n\nBASE = "Alibaba-NLP/gte-multilingual-reranker-base"\nMAX_LEN, NEG = 512, 7\n\n\ndef ndcg5(rels):\n    dcg = sum((2 ** r - 1) / math.log2(i + 2) for i, r in enumerate(rels[:5]))\n    ideal = sorted(rels, reverse=True)\n    idcg = sum((2 ** r - 1) / math.log2(i + 2) for i, r in enumerate(ideal[:5]))\n    return dcg / idcg if idcg else None\n\n\ndef mrr5(rels):\n    return next((1 / (i + 1) for i, r in enumerate(rels[:5]) if r == 2), 0.0)\n\n\ndef scores_of(groups, order_fn):\n    rows = []\n    for g in groups:\n        order = order_fn(g)\n        rels = [p["relevance"] for p in order]\n        if max(rels) == 0:\n            continue\n        rows.append({"company": g["company"], "ndcg5": ndcg5(rels), "mrr5": mrr5(rels),\n                     "hit5": float(any(r == 2 for r in rels[:5])) if 2 in rels else None})\n    def mean(k):\n        def f(rs):\n            v = [r[k] for r in rs if r[k] is not None]\n            return sum(v) / len(v) if v else 0.0\n        return f\n    comp = [r["company"] for r in rows]\n    return {k: {"mean": round(mean(k)(rows), 4), "ci95": boot_ci(rows, comp, mean(k))} for k in ("ndcg5", "mrr5", "hit5")} | \\\n        {"claims": len(rows)}\n\n\ndef system_order(name):\n    def f(g):\n        def rank(p):\n            s = p["systems"]\n            return s.get(name, 99) if isinstance(s, dict) else 99\n        return sorted(g["pairs"], key=lambda p: (rank(p), p["id"]))\n    return f\n\n\ndef load_model(path):\n    import torch\n    from transformers import AutoModelForSequenceClassification, AutoTokenizer\n    dev = "cuda" if torch.cuda.is_available() else "cpu"\n    tok = AutoTokenizer.from_pretrained(path, trust_remote_code=True)\n    model = AutoModelForSequenceClassification.from_pretrained(path, trust_remote_code=True, num_labels=1).float().to(dev)\n    return model, tok, dev\n\n\ndef score_pairs(model, tok, dev, claim, passages):\n    import torch\n    with torch.no_grad():\n        x = tok([claim] * len(passages), passages, truncation=True, max_length=MAX_LEN, padding=True,\n                return_tensors="pt").to(dev)\n        return model(**x).logits.view(-1).float().cpu().tolist()\n\n\ndef model_order(model, tok, dev):\n    def f(g):\n        s = score_pairs(model, tok, dev, g["claim"], [p["passage"] for p in g["pairs"]])\n        return [p for _, p in sorted(zip(s, g["pairs"]), key=lambda t: -t[0])]\n    return f\n\n\ndef fit(train_groups, epochs, lr, seed, hist, tag):\n    import torch\n    from transformers import get_linear_schedule_with_warmup\n    seed_all(seed)\n    model, tok, dev = load_model(BASE)\n    items = []\n    for g in train_groups:\n        pos = [p for p in g["pairs"] if p["relevance"] == 2]\n        neg = [p for p in g["pairs"] if p["relevance"] < 2]\n        for p in pos:\n            if neg:\n                items.append((g["claim"], p, neg))\n    steps = max(1, epochs * len(items))\n    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)\n    sch = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)\n    model.train()\n    for ep in range(epochs):\n        random.shuffle(items)\n        tot = 0.0\n        for claim, pos, neg in items:\n            cand = [pos] + random.sample(neg, min(NEG, len(neg)))\n            x = tok([claim] * len(cand), [c["passage"] for c in cand], truncation=True, max_length=MAX_LEN,\n                    padding=True, return_tensors="pt").to(dev)\n            with torch.autocast(dev, dtype=torch.float16, enabled=dev == "cuda"):\n                z = model(**x).logits.view(1, -1).float()\n            loss = torch.nn.functional.cross_entropy(z, torch.zeros(1, dtype=torch.long, device=dev))\n            opt.zero_grad()\n            loss.backward()\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n            opt.step()\n            sch.step()\n            tot += loss.item()\n        hist.append({"run": tag, "epoch": ep + 1, "loss": round(tot / max(1, len(items)), 4), "items": len(items)})\n        log(f"[rerank] {tag} epoch {ep + 1} loss {tot / max(1, len(items)):.4f} ({len(items)} lists)")\n    return model.eval(), tok, dev\n\n\ndef main() -> int:\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--epochs", type=int, default=3)\n    ap.add_argument("--lr", type=float, default=2e-5)\n    ap.add_argument("--folds", type=int, default=5)\n    args = ap.parse_args()\n    import torch\n\n    cell("rerank data")\n    lab = {x["id"]: x for x in jl("pairs_labels.jsonl")}\n    by = defaultdict(list)\n    for p in jl("pairs_pool.jsonl"):\n        if p["id"] in lab and not p["overlaps_gs_eval"]:\n            s = p.get("systems")\n            if isinstance(s, str):\n                try:\n                    s = ast.literal_eval(s)\n                except (ValueError, SyntaxError):\n                    s = {}\n            by[p["claim_id"]].append({**p, "systems": s or {}, "relevance": lab[p["id"]]["relevance"]})\n    groups = [{"claim_id": c, "claim": ps[0]["claim"], "company": ps[0]["company_id"], "origin": ps[0]["origin"],\n               "pairs": ps} for c, ps in by.items()]\n    lib = [g for g in groups if g["origin"] == "lib"]\n    log(f"[rerank] claims {len(groups)} (lib {len(lib)}), pairs {sum(len(g[\'pairs\']) for g in groups)}")\n    od = out_dir("reranker")\n    m, hist = {"claims": len(groups), "lib_claims": len(lib)}, []\n\n    cell("rerank baselines")\n    m["lite"] = scores_of(lib, system_order("lite"))\n    m["bm25"] = scores_of(lib, system_order("bm25"))\n    zs, ztok, zdev = load_model(BASE)\n    zs.eval()\n    m["zero_shot"] = scores_of(lib, model_order(zs, ztok, zdev))\n    del zs\n    torch.cuda.empty_cache()\n    log(f"[rerank] lite {m[\'lite\'][\'ndcg5\']} · zero-shot {m[\'zero_shot\'][\'ndcg5\']}")\n\n    cell("rerank cv")\n    t = Timer()\n    comp = [g["company"] for g in groups]\n    fold_scores = []\n    for f, (tr, te) in enumerate(company_folds(comp, args.folds)):\n        model, tok, dev = fit([groups[i] for i in tr], args.epochs, args.lr, 13, hist, f"fold{f}")\n        test = [groups[i] for i in te if groups[i]["origin"] == "lib"]\n        order = model_order(model, tok, dev)\n        for g in test:\n            fold_scores.append({**g, "pairs": order(g)})\n        del model\n        torch.cuda.empty_cache()\n    m["fine_tuned_cv"] = scores_of(fold_scores, lambda g: g["pairs"])\n    m["cv_seconds"] = t()\n    log(f"[rerank] fine-tuned CV {m[\'fine_tuned_cv\'][\'ndcg5\']}")\n\n    cell("rerank final + cpu latency")\n    model, tok, dev = fit(groups, args.epochs, args.lr, 13, hist, "final")\n    model.save_pretrained(od / "gte-reranker-final")\n    tok.save_pretrained(od / "gte-reranker-final")\n    cpu = model.to("cpu")\n    torch.set_num_threads(4)\n    g = lib[0]\n    t0 = time.time()\n    score_pairs(cpu, tok, "cpu", g["claim"], [p["passage"] for p in g["pairs"]][:10])\n    m["cpu_ms_per_passage_4threads"] = round((time.time() - t0) / min(10, len(g["pairs"])) * 1000, 1)\n    save_json(od / "metrics.json", m)\n    save_json(od / "history.json", hist)\n    log(f"[rerank] cpu {m[\'cpu_ms_per_passage_4threads\']} ms/passage · DONE")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n',
}
for name, text in FILES.items():
    open(name, 'w', encoding='utf-8').write(text)
    print('wrote', name, len(text), flush=True)


In [ ]:
print('[cell] run 1: python -u train_stance_enc.py', flush=True)
import subprocess, sys, os
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
sh('python -u train_stance_enc.py')


In [ ]:
print('[cell] run 2: python -u train_reranker.py', flush=True)
import subprocess, sys, os
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
sh('python -u train_reranker.py')


In [ ]:
print("[cell] done", flush=True)
print("ALL CELLS OK", flush=True)